# NLP Tweets - GPT2 Probe
```
GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)           # token embeddings
    (wpe): Embedding(1024, 768)            # positional embeddings
    (drop): Dropout(p=0.1)                 # input dropout
    (h): ModuleList([                      # 12× GPT2Block
      GPT2Block(
        (ln_1): LayerNorm(768)             # pre‐attention layer norm
        (attn): GPT2Attention(             # multi‑head self‑attention
          (c_attn): Conv1D(2304, 768)      # QKV projection
          (c_proj): Conv1D(768, 768)       # output projection
          (attn_dropout): Dropout(0.1)
          (resid_dropout): Dropout(0.1)
        )
        (ln_2): LayerNorm(768)             # post‑attention layer norm
        (mlp): GPT2MLP(                    # feed‑forward MLP
          (c_fc): Conv1D(3072, 768)        # first FC
          (c_proj): Conv1D(768, 3072)      # second FC
          (act): NewGELUActivation()       # GELU
          (dropout): Dropout(0.1)
        )
      )
      …  # repeated 12 times
    ])
    (ln_f): LayerNorm(768)                 # final layer norm
  )
  (score): Linear(768 → num_labels)        # your classification head
)
```

In [ ]:
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
import transformers 
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model

In [ ]:
id2label={0: "NEGATIVE",    1: "POSITIVE"}     
label2id={   "NEGATIVE": 0,    "POSITIVE": 1}

model_name = "gpt2"
model      = GPT2ForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,
    id2label=id2label, 
    label2id=label2id,
)
model.config.pad_token_id = model.config.eos_token_id
model.to(device)
base_model = model


tokenizer  = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})


# Parameter Efficient Fine-Tuning - only train probe head, not base model
for param in model.parameters():            param.requires_grad = True
for param in model.base_model.parameters(): param.requires_grad = False
model

# Pandas Dataframes

In [ ]:
!ls -la /kaggle/input/nlp-getting-started

In [ ]:
from sklearn.model_selection import train_test_split

train_df_csv         = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test_df              = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
sample_submission_df = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')

train_df, eval_df = train_test_split(train_df_csv, test_size=0.2, stratify=train_df_csv['target'], random_state=42)

# Huggingface Datasets

In [ ]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train_df)
eval_dataset  = Dataset.from_pandas(eval_df)
test_dataset  = Dataset.from_pandas(test_df)

# Tokenize in batch mode (recommended)
def tokenize_function(batch):
    tokenized = tokenizer(batch["text"], padding="max_length", truncation=True)
    if 'target' in batch:
        tokenized['labels'] = batch['target']
        # tokenized["labels"] = [float(x) for x in batch["target"]] # Convert ints to floats for MSELoss
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval  =  eval_dataset.map(tokenize_function, batched=True)
tokenized_test  =  test_dataset.map(tokenize_function, batched=True)

tokenized_dataset = DatasetDict({
    "train": tokenized_train,
    "eval":  tokenized_eval,
    "test":  tokenized_test,
})

# Model

In [ ]:
%%time
# LINT-FIX: You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
tokenizer.deprecation_warnings["Asking-to-pad-a-fast-tokenizer"] = True

import numpy as np
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        "accuracy": (predictions == labels).mean()
    }


# The HuggingFace Trainer class handles the training and eval loop for PyTorch for us.
# Read more about it here https://huggingface.co/docs/transformers/main_classes/trainer
trainer = Trainer(
    model = model,
    args  = TrainingArguments(
        output_dir = "./data/sentiment_analysis",
        # learning_rate = 2e-3,              # Udacity
        learning_rate = 5e-4,               # ChatGPT https://chatgpt.com/c/680609fe-cbd4-8004-a8c2-a069a1e57741
        lr_scheduler_type = "linear",       # ChatGPT https://chatgpt.com/c/680609fe-cbd4-8004-a8c2-a069a1e57741
        warmup_steps = 100,                 # ChatGPT https://chatgpt.com/c/680609fe-cbd4-8004-a8c2-a069a1e57741
        
        per_device_train_batch_size = 4,  # Reduce the batch size if you don't have enough memory
        per_device_eval_batch_size  = 4,
        num_train_epochs = 25 if (os.environ.get('KAGGLE_KERNEL_RUN_TYPE','') == 'Batch') else 1,
        weight_decay = 0.01,
        # evaluation_strategy="epoch",
        # save_strategy="epoch",
        # load_best_model_at_end=True,
        report_to="none"
    ),
    train_dataset   = tokenized_dataset["train"],
    eval_dataset    = tokenized_dataset["eval"],
    tokenizer       = tokenizer,
    data_collator   = DataCollatorWithPadding(tokenizer=tokenizer), 
    compute_metrics = compute_metrics,
)

In [ ]:
# %%time
# # Untrained Probe Head
# print("START trainer.evaluate()")
# trainer.evaluate()
# print("END   trainer.evaluate()")

In [ ]:
# print("START trainer.train()")
# trainer.train()
# # print("END   trainer.train()")

In [ ]:
# %%time
# # Trained Probe Head
# print("START trainer.evaluate()")
# trainer.evaluate()
# # print("END   trainer.evaluate()")

# PEFT

In [ ]:
from peft import get_peft_model, LoraConfig

# 2) Configure LoRA—only inject trainable rank‑r matrices into attention & MLP
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn","c_proj"],  # you can also target "mlp.c_fc", etc.
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
)

# 3) Wrap model
model = get_peft_model(base_model, lora_cfg)
model

In [ ]:
print("START trainer.train()")
trainer.train()
# print("END   trainer.train()")

In [ ]:
%%time
# PEFT + Trained Probe Head
print("START trainer.evaluate()")
trainer.evaluate()
# print("END   trainer.evaluate()")

# Save
- https://chatgpt.com/g/g-p-67fe3cc81cc4819181ea2f77bbffd7e8-generative-ai-nanodegree/c/680609fe-cbd4-8004-a8c2-a069a1e57741

In [ ]:
output_dir = "GPT2_tweets_probe"
trainer.model.save_pretrained(output_dir)      # 1. Save the model’s weights + config
trainer.tokenizer.save_pretrained(output_dir)  # 2. Save the tokenizer (important if you changed padding tokens, special tokens…)
trainer.state.save_to_json(os.path.join(output_dir, "trainer_state.json"))  # 3. (Optional) Save TrainingArguments & Trainer state
# Note: you can also pickle the full Trainer if you really want to resume training exactly.


# Reload
- https://chatgpt.com/g/g-p-67fe3cc81cc4819181ea2f77bbffd7e8-generative-ai-nanodegree/c/680609fe-cbd4-8004-a8c2-a069a1e57741

In [ ]:
from transformers import GPT2ForSequenceClassification, GPT2Tokenizer, Trainer, TrainingArguments

output_dir = "GPT2_tweets_probe"
model     = GPT2ForSequenceClassification.from_pretrained(output_dir, num_labels=2)          # 1. Load the model & config
tokenizer = GPT2Tokenizer.from_pretrained(output_dir)                                        # 2. Load the tokenizer
trainer.state = trainer.state.from_json_file(os.path.join(output_dir, "trainer_state.json")) # 3. rerun cell [7] first

# # 3. (Optional) Reconstruct your Trainer if you want to resume training
# args = TrainingArguments(
#     output_dir="./resume_dir",
#     # … your original args …
# )
# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     tokenizer=tokenizer,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics,
# )
# # You can restore the Trainer state if you saved it:
# trainer.state = trainer.state.from_json_file(os.path.join(load_dir, "trainer_state.json"))


# Sentiment Pipeline

In [ ]:
%%time
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model=model, 
    tokenizer=tokenizer, 
    padding="max_length", 
    truncation=True, 
    device=device
)
print(sentiment_pipeline("Everything is Awesome!"), '->', "Everything is Awesome!")
print(sentiment_pipeline("Everything is Bogus!"),   '->', "Everything is Bogus!")

# Submission.csv

In [ ]:
%%time
output = trainer.predict(tokenized_test)
logits = output.predictions        # shape (n_samples, num_labels) or (n_samples,)
preds  = np.argmax(logits, axis=1) # 4. Convert logits → {0,1} targets If you used a 2‑class head (num_labels=2):

submission = pd.DataFrame({
    "id":     tokenized_dataset["test"]["id"],
    "target": preds
})

submission.to_csv("submission.csv", index=False)